In [33]:
from IPython.display import display
import numpy as np
import pandas as pd
from scipy import stats

import main_analysis as ma

RESULTS_DIR = 'results'

# Section 2.2

In [42]:
SUBJECTSUBJECT_CORRELATIONS_CSV_PATH = f'{RESULTS_DIR}/subjectsubject_correlations_nsd515.csv'
MODELSUBJECT_CORRELATIONS_CSV_PATH = f'{RESULTS_DIR}/modelsubject_correlations_nsd515.csv'

core_frois = ['FFA', 'PPA', 'EBA']
tthreshold = 7

to_model_names = ma.get_model_names('task_optimized')
model_localizer = 'grillspector'
best_model_name = 'vitb16siglip2res224'

metrics = ['normed_pearsonr', 'normed_rdmr']

## Human-Human

In [69]:
hh_csv = pd.read_csv(SUBJECTSUBJECT_CORRELATIONS_CSV_PATH)
hh_csv = hh_csv[
    (hh_csv['fmri_froi'].isin(core_frois)) &
    (hh_csv['fmri_tthreshold'] == tthreshold)
]
print(f'Minimum normed pearsonr across core ROIs and across all subject pairs:')
print(hh_csv['normed_pearsonr'].min().item())
print(f'Maximum normed pearsonr p-value across core ROIs and across all subject pairs:')
print(hh_csv['pvalue_pearsonr'].max().item())
print(f'Minimum normed rdmr across core ROIs and across all subject pairs:')
print(hh_csv['normed_rdmr'].min().item())
print(f'Maximum normed rdmr p-value across core ROIs and across all subject pairs:')
print(hh_csv['pvalue_rdmr'].max().item())
# hh_csv_grouped = hh_csv.groupby(['fmri_froi']).aggregate(
#     {
#         'normed_pearsonr': ['median'],
#         'normed_rdmr': ['median'],
#     }
# )
hh_csv_grouped = (
    hh_csv
    .groupby(['fmri_froi'])
    .agg(
        normed_pearsonr_median=('normed_pearsonr', 'median'),
        normed_rdmr_median=('normed_rdmr', 'median'),
    )
    .reset_index()
).set_index('fmri_froi')
display(hh_csv_grouped)
print('Median normed pearsonr across core ROIs:')
print(hh_csv_grouped['normed_pearsonr_median'].median().item())
print('Median normed rdmr across core ROIs:')
print(hh_csv_grouped['normed_rdmr_median'].median().item())

Minimum normed pearsonr across core ROIs and across all subject pairs:
0.8727959208707898
Maximum normed pearsonr p-value across core ROIs and across all subject pairs:
1.4775544305207639e-38
Minimum normed rdmr across core ROIs and across all subject pairs:
0.8050828196815112
Maximum normed rdmr p-value across core ROIs and across all subject pairs:
0.0


,normed_pearsonr_median,normed_rdmr_median
fmri_froi,,
EBA,0.997637,1.127444
FFA,0.983508,1.009670
PPA,1.014474,1.069014


Median normed pearsonr across core ROIs:
0.997636511462672
Median normed rdmr across core ROIs:
1.0690142237039242


## Model-Human

In [68]:
mh_csv = pd.read_csv(MODELSUBJECT_CORRELATIONS_CSV_PATH)
mh_csv = mh_csv[
    (mh_csv['fmri_froi'].isin(core_frois)) &
    (mh_csv['fmri_tthreshold'] == tthreshold) &
    (mh_csv['model_name'].isin(to_model_names)) &
    (mh_csv['model_localizer'] == model_localizer) &
    (mh_csv['model_froi'].isin(core_frois)) &
    (mh_csv['model_tthreshold'] == tthreshold)
]



mh_csv_grouped = (
    mh_csv
    .groupby(['fmri_froi', 'model_name'])
    .agg(
        normed_pearsonr_median=('normed_pearsonr', 'median'),
        normed_rdmr_median=('normed_rdmr', 'median'),
    )
    .reset_index()
)

mh_csv_grouped_best = mh_csv_grouped[
    mh_csv_grouped['model_name'] == best_model_name
]
# drop the model_name column & set fmri_froi as index
mh_csv_grouped_best = mh_csv_grouped_best.drop(columns=['model_name']).set_index('fmri_froi')
print(f'Model-Subject correlations for best model: {best_model_name}')
display(mh_csv_grouped_best)

mh_csv_grouped2 = (
    mh_csv_grouped
    .groupby('fmri_froi')
    .agg(
        normed_pearsonr_median=('normed_pearsonr_median', 'median'),
        normed_rdmr_median=('normed_rdmr_median', 'median'),
    )
    .reset_index()
).set_index('fmri_froi')
print('Model-Subject correlations (grouped by ROI):')
display(mh_csv_grouped2)

Model-Subject correlations for best model: vitb16siglip2res224


,normed_pearsonr_median,normed_rdmr_median
fmri_froi,,
EBA,0.764139,0.481560
FFA,0.657793,0.574184
PPA,0.711801,0.310059


Model-Subject correlations (grouped by ROI):


,normed_pearsonr_median,normed_rdmr_median
fmri_froi,,
EBA,0.672800,0.306611
FFA,0.618296,0.383438
PPA,0.726980,0.367889


### ANN-brain < brain-brain corrs

In [43]:
pvalues_dict = {}
model_medians_dict = {}
for metric in metrics:
    metric_min_pval = np.inf
    pvalues_dict[metric] = {}
    model_medians_dict[metric] = {}
    print(f'For metric: {metric}')
    for froi in core_frois:
        pvalues_dict[metric][froi] = {}
        model_medians_dict[metric][froi] = {}
        print(f'  For ROI: {froi}')
        hh_subset = hh_csv[
            (hh_csv['fmri_froi'] == froi)
        ]
        hh_corrs = hh_subset[metric].values
        for model_name in to_model_names:
            subset = mh_csv[
                (mh_csv['fmri_froi'] == froi) &
                (mh_csv['model_name'] == model_name)
            ]
            mh_corrs = subset[metric].values
            u_stat, p_value = stats.mannwhitneyu(
                mh_corrs,
                hh_corrs,
                alternative='less'
            )
            if p_value >= 0.05:
                print(f'    Model {model_name} p-value: {p_value} (NOT SIGNIFICANT)')
            # else:
            #     print(f'    Model {model_name} p-value: {p_value} (SIGNIFICANT)')
            pvalues_dict[metric][froi][model_name] = p_value
            model_medians_dict[metric][froi][model_name] = np.median(mh_corrs)
            if p_value < metric_min_pval:
                metric_min_pval = p_value
        metric_froi_best_model_median = np.median(model_medians_dict[metric][froi][best_model_name])
        metric_froi_model_medians = model_medians_dict[metric][froi].values()
        median_of_medians = np.median(list(metric_froi_model_medians))
        print(f'    Median of model medians for ROI {froi} and metric {metric}: {median_of_medians}')
        # Print the median for the best model
        print(f'    Median of {best_model_name} for ROI {froi} and metric {metric}: {metric_froi_best_model_median}')
    print('\n')
    print(f'Minimum p-value for metric {metric} across core ROIs and models: {metric_min_pval}')
    print('\n')

For metric: normed_pearsonr
  For ROI: FFA
    Median of model medians for ROI FFA and metric normed_pearsonr: 0.6182962524942974
    Median of vitb16siglip2res224 for ROI FFA and metric normed_pearsonr: 0.6577928973056599
  For ROI: PPA
    Median of model medians for ROI PPA and metric normed_pearsonr: 0.7269798643138887
    Median of vitb16siglip2res224 for ROI PPA and metric normed_pearsonr: 0.7118005729011971
  For ROI: EBA
    Median of model medians for ROI EBA and metric normed_pearsonr: 0.6727995765455339
    Median of vitb16siglip2res224 for ROI EBA and metric normed_pearsonr: 0.7641392044521946


Minimum p-value for metric normed_pearsonr across core ROIs and models: 3.3046555326212464e-08


For metric: normed_rdmr
  For ROI: FFA
    Median of model medians for ROI FFA and metric normed_rdmr: 0.3834384870617388
    Median of vitb16siglip2res224 for ROI FFA and metric normed_rdmr: 0.5741843944220171
  For ROI: PPA
    Median of model medians for ROI PPA and metric normed_rdmr

In [71]:
# Subtract the grouped model-subject correlations from the grouped subject-subject correlations
print('GAP between subject-subject and model-subject correlations (grouped by ROI):')
median_gap_df = hh_csv_grouped - mh_csv_grouped2
display(median_gap_df)

print(f'GAP between subject-subject and model-subject correlations for the best model {best_model_name} (grouped by ROI):')
best_gap_df = hh_csv_grouped - mh_csv_grouped_best
print(best_gap_df)

GAP between subject-subject and model-subject correlations (grouped by ROI):


,normed_pearsonr_median,normed_rdmr_median
fmri_froi,,
EBA,0.324837,0.820833
FFA,0.365212,0.626231
PPA,0.287494,0.701125


GAP between subject-subject and model-subject correlations for the best model vitb16siglip2res224 (grouped by ROI):
           normed_pearsonr_median  normed_rdmr_median
fmri_froi                                            
EBA                      0.233497            0.645884
FFA                      0.325715            0.435485
PPA                      0.302673            0.758955


In [21]:
mh_csv

,model_name,model_layer,model_froi,model_tthreshold,model_localizer,fmri_dataset,fmri_froi,fmri_tthreshold,subject,normed_pearsonr,pearsonr,normed_rdmr,rdmr,pvalue_rdmr,pvalue_pearsonr,fmri_splithalf_pearsonr,fmri_splithalf_rdmr
0,alexnetimagenet1kv1,features.11,FFA,7,grillspector,nsd515,FFA,7,1,0.579079,0.524384,0.356567,0.264244,0.0,1.011229e-37,0.820017,0.549200
2,alexnetimagenet1kv1,features.12,EBA,7,grillspector,nsd515,EBA,7,1,0.509453,0.457236,0.259660,0.194928,0.0,5.709499e-28,0.805513,0.563560
4,alexnetimagenet1kv1,features.12,PPA,7,grillspector,nsd515,PPA,7,1,0.619488,0.538884,0.198041,0.131808,0.0,3.969849e-40,0.756702,0.442968
7,alexnetimagenet1kv1,features.11,FFA,7,grillspector,nsd515,FFA,7,2,0.657376,0.597486,0.502901,0.380867,0.0,3.774516e-51,0.826090,0.573563
9,alexnetimagenet1kv1,features.12,EBA,7,grillspector,nsd515,EBA,7,2,0.496988,0.431335,0.269299,0.192396,0.0,9.474955e-25,0.753248,0.510411
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1948,convnexttinyimagenet1kv1,features.5.7.block.3,EBA,7,grillspector,nsd515,EBA,7,7,0.724770,0.612722,0.519005,0.346175,0.0,2.113391e-54,0.714703,0.444887
1950,convnexttinyimagenet1kv1,features.5.2.block.3,PPA,7,grillspector,nsd515,PPA,7,7,0.806977,0.595580,0.595779,0.281019,0.0,9.363155e-51,0.544700,0.222485
1953,convnexttinyimagenet1kv1,features.5.7.block.3,FFA,7,grillspector,nsd515,FFA,7,8,0.627813,0.505135,0.472118,0.276250,0.0,1.058845e-34,0.647372,0.342375
1955,convnexttinyimagenet1kv1,features.5.7.block.3,EBA,7,grillspector,nsd515,EBA,7,8,0.724700,0.516433,0.671391,0.305870,0.0,1.886664e-36,0.507821,0.207550


In [20]:
35*8*3

840